# FitGuard 当前跟踪版双模态数据采集

本页同步采集当前人体跟踪特征与无线腰部 IMU，只写入 `FitGuard_Unified_Test/dataset_motion_v2`。每次使用新的受试者、类别和试次编号，已有试次不会被覆盖。

In [ ]:
from pathlib import Path
import hashlib
import importlib
import sys
import cv2
import ipywidgets as widgets
from IPython.display import display
base = (Path.cwd().parent / 'FitGuard_Unified_Test').resolve()
deploy = Path.cwd().resolve()
# 强制当前部署目录优先，防止导入基础工程中的旧同名采集器。
for path in (str(deploy), str(base)):
    while path in sys.path:
        sys.path.remove(path)
sys.path[:0] = [str(deploy), str(base)]
sys.modules.pop('fitguard_dual_collector_gateway', None)
sys.modules.pop('motion_tracker_v2', None)
import fitguard_dual_collector_gateway as collector_module
collector_module = importlib.reload(collector_module)
assert Path(collector_module.__file__).resolve().parent == deploy, collector_module.__file__
tracker_module = importlib.import_module('motion_tracker_v2')
tracker_path = Path(tracker_module.__file__).resolve()
assert tracker_path.parent == deploy, tracker_path
collect_trial_gateway = collector_module.collect_trial_gateway
print('当前采集器：', collector_module.__file__)
print('当前跟踪器：', tracker_path)
print('跟踪器指纹：', hashlib.sha256(tracker_path.read_bytes()).hexdigest()[:16])

# 第一次先保持 TEST_MODE=True 做10秒检查；检查通过后改为 False。
TEST_MODE = False
SUBJECT = 'M06'
LABEL = 'Marked Abnormality'  # Stable / Mild Abnormality / Marked Abnormality
TRIAL = '03'
SECONDS = 10 if TEST_MODE else 30
OUTPUT_NAME = 'dataset_motion_v2_check' if TEST_MODE else 'dataset_motion_v2'
print('运行模式：', '10秒链路检查（不进入训练集）' if TEST_MODE else '30秒正式采集')

preview = widgets.Image(format='jpeg', width=960)
status = widgets.HTML(value='<b>准备采集，请点击运行后先保持静止完成校准。</b>')
display(widgets.VBox([status, preview]))
preview_state = {'first_timestamp': None}

def show_preview(annotated, motion_mask, record):
    if preview_state['first_timestamp'] is None:
        preview_state['first_timestamp'] = record.timestamp
    elapsed = max(0.0, record.timestamp - preview_state['first_timestamp'])
    remaining = max(0.0, SECONDS - elapsed)
    ok, encoded = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 78])
    if ok:
        preview.value = encoded.tobytes()
    state = 'LOCKED' if record.valid else 'LOST'
    color = '#16803a' if record.valid else '#c62828'
    quality = float(getattr(record, 'quality', 0.0))
    source = getattr(record, 'source', 'legacy-feature')
    status.value = (f'<b style="color:{color}">{state}</b> &nbsp; '                    f'Q={quality:.2f} &nbsp; {source} &nbsp; '                    f'剩余约 {remaining:.1f} 秒')

summary = collect_trial_gateway(
    subject=SUBJECT,
    label=LABEL,
    seconds=SECONDS,
    trial=TRIAL,
    bitstream=str(base / 'fitguard_unified.bit'),
    output_root=str(base / OUTPUT_NAME),
    port='/dev/ttyUSB0',
    preview_callback=show_preview,
    preview_interval=0.50,
    training_candidate=not TEST_MODE,
    min_visual_hz=10.0,
)
if TEST_MODE:
    status.value = '<b style="color:#1565c0">短测完成：只用于检查，不进入训练集。</b>'
elif summary['accepted_for_training']:
    status.value = '<b style="color:#16803a">正式采集通过质量门槛，已进入训练集。</b>'
else:
    failed = [name for name, ok in summary['quality_checks'].items() if not ok]
    status.value = '<b style="color:#c62828">本次未进入训练集：</b>' + ', '.join(failed)
summary